In [1]:
import numpy as np
import pandas as pd


In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data"

columns = [
    "Id","RI","Na","Mg","Al","Si","K","Ca","Ba","Fe","Type"
]

df = pd.read_csv(url, header=None, names=columns)
df


,Id,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,1
1,2,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,1
2,3,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,1
3,4,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,1
4,5,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...
209,210,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,7
210,211,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,7
211,212,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,7
212,213,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,7


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


Number of rows: 214
Number of columns: 11


In [4]:
df.head()
#

,Id,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,2,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,3,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,4,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,5,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [6]:
# in glass 7 classes convert to binary 0 or 1 wala

df["Type"] = df["Type"].apply(lambda x: 1 if x == 1 else 0)


In [8]:
# separate features and target
X = df.drop(["Id", "Type"], axis=1).values
y = df["Type"].values


In [9]:
# feature normalization

def normalize(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)

X = normalize(X)


In [11]:
# train test validation split

def split_data(X, y, train_size=0.7, val_size=0.15):
    m = len(X)
    train_end = int(train_size * m)
    val_end = train_end + int(val_size * m)

    X_train = X[:train_end]
    y_train = y[:train_end]

    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]

    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = split_data(X, y)


In [12]:
# sigmoid fxn

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


In [13]:
# initialize parameters

def initialize_parameters(n_features):
    weights = np.zeros(n_features)
    bias = 0
    return weights, bias


In [14]:
# forward propagation

def forward(X, weights, bias):
    z = np.dot(X, weights) + bias
    y_hat = sigmoid(z)
    return y_hat


In [15]:
# Loss Function (Log Loss)

def compute_loss(y, y_hat):
    epsilon = 1e-9
    loss = -np.mean(y * np.log(y_hat + epsilon) + (1 - y) * np.log(1 - y_hat + epsilon))
    return loss


In [16]:
# Backward Propagation
def backward(X, y, y_hat):
    m = len(y)
    dw = np.dot(X.T, (y_hat - y)) / m
    db = np.mean(y_hat - y)
    return dw, db


In [17]:
# training function

def train(X, y, lr=0.01, epochs=1000):
    weights, bias = initialize_parameters(X.shape[1])

    for i in range(epochs):
        y_hat = forward(X, weights, bias)
        loss = compute_loss(y, y_hat)

        dw, db = backward(X, y, y_hat)

        weights -= lr * dw
        bias -= lr * db

        if i % 100 == 0:
            print(f"Epoch {i}, Loss: {loss:.4f}")

    return weights, bias


In [18]:
# validation fxn

def validate(X, y, weights, bias):
    y_hat = forward(X, weights, bias)
    predictions = (y_hat >= 0.5).astype(int)
    accuracy = np.mean(predictions == y)
    return accuracy


In [19]:
# test fxn

def test(X, y, weights, bias):
    y_hat = forward(X, weights, bias)
    predictions = (y_hat >= 0.5).astype(int)
    accuracy = np.mean(predictions == y)
    return accuracy


In [20]:
# train model

weights, bias = train(X_train, y_train, lr=0.01, epochs=1000)


Epoch 0, Loss: 0.6931
Epoch 100, Loss: 0.6702
Epoch 200, Loss: 0.6535
Epoch 300, Loss: 0.6406
Epoch 400, Loss: 0.6301
Epoch 500, Loss: 0.6213
Epoch 600, Loss: 0.6137
Epoch 700, Loss: 0.6070
Epoch 800, Loss: 0.6012
Epoch 900, Loss: 0.5959


In [21]:
# validation accuracy

val_accuracy = validate(X_val, y_val, weights, bias)
print("Validation Accuracy:", val_accuracy)


Validation Accuracy: 0.6875


In [23]:
# test accuracy

test_accuracy = test(X_test, y_test, weights, bias)
print("Test Accuracy:", test_accuracy)


Test Accuracy: 0.8787878787878788
